# **Fetching additional energy variables and merging with the  existing mastersets**
**WU Vienna | SBWL Data Science | DSLab 2026S | Marbl Energy**

*Authors: Pavle Cvijanovic, Aleksandar Gradev, Aleksandar Milosavljevic, Vincent Skakala*

### Goal
The purpose of this notebook was to error-free fetch different variables from ENTSO-E API across 5 different bidding zones we are testing. After this hase been done successfully, fethed fariables will be joined with the other energy price featured based on date. These features come from a separate datasets, created in the "03_commodity_merge.ipynb" notebook. Merged data is then stored on desktop. 

Setting up the API pipeline:

In [ ]:
from entsoe import EntsoePandasClient
import pandas as pd
TOKEN = "API-KEY" #enter your sectret key
client = EntsoePandasClient(TOKEN)

Defining our time interval and bidding zone codes

In [2]:
start = pd.Timestamp('20230101', tz='Europe/Brussels')
end = pd.Timestamp('20260329', tz='Europe/Brussels')

countries = {
    'FR': 'France',
    'DE_LU': 'Germany',
    'ES': 'Spain',
    'DK_1': 'Denmark 1',
    'NO_2': 'Norway 2',
}

# Section 1 - Fetching additional data
### Downloading new variables from ENTSO-E
Creating a custom function to automatically download 5 different energy variables for every bidding zone. It will say if a column has been succesfully downloaded or if it is returning empty values

In [ ]:
def fetch_all_data(client, country_code, start, end):
    results = {}

    fetchers = {
        'load_forecast':            lambda: client.query_load_forecast(country_code, start=start, end=end),
        'wind_solar_forecast':      lambda: client.query_wind_and_solar_forecast(country_code, start=start, end=end),
        'generation_forecast':      lambda: client.query_generation_forecast(country_code, start=start, end=end),
        'hydro_reservoir':          lambda: client.query_aggregate_water_reservoirs_and_hydro_storage(country_code, start=start, end=end),
        'unavailability_gen_units': lambda: client.query_unavailability_of_generation_units(country_code, start=start, end=end, docstatus='A26'),
    }

    for name, func in fetchers.items():
        try:
            results[name] = func()
            print(f"  ✓ {name}")
        except Exception as e:
            print(f"  ✗ {name}: {e}")
            results[name] = None

    return results


Fetching data for France (FR)...
  ✓ load_forecast
  ✓ wind_solar_forecast
  ✓ generation_forecast
  ✓ hydro_reservoir
  ✗ unavailability_gen_units: 400 Client Error:  for url: https://web-api.tp.entsoe.eu/api?documentType=A80&biddingZone_domain=10YFR-RTE------C&offset=0&docStatus=A26&securityToken=4896cc0f-37da-49ba-a3f1-387906966051&periodStart=202212312300&periodEnd=202312312300

Fetching data for Germany (DE_LU)...
  ✓ load_forecast
  ✓ wind_solar_forecast
  ✓ generation_forecast
  ✗ hydro_reservoir: 
  ✗ unavailability_gen_units: 400 Client Error:  for url: https://web-api.tp.entsoe.eu/api?documentType=A80&biddingZone_domain=10Y1001A1001A82H&offset=0&docStatus=A26&securityToken=4896cc0f-37da-49ba-a3f1-387906966051&periodStart=202212312300&periodEnd=202312312300

Fetching data for Spain (ES)...
  ✓ load_forecast
  ✓ wind_solar_forecast
  ✓ generation_forecast
  ✓ hydro_reservoir
  ✗ unavailability_gen_units: 400 Client Error:  for url: https://web-api.tp.entsoe.eu/api?documentType

Downloading the additional features:

In [ ]:
all_data = {}
for code, name in countries.items():
    print(f"\nFetching data for {name} ({code})...")
    all_data[code] = fetch_all_data(client, code, start, end)

Quickly checking some of the columns

In [ ]:
all_data['FR']['wind_solar_forecast']       # France load forecast as a DataFrame/Series
#all_data['DE_LU']['wind_solar_forecast']  # Germany wind & solar forecast

# Section 2 - Merging
### Loading the mastersets and merging them with fetcher variables
Now we are going to create a function that will merge multiple time series datasets for a single country into one wide DataFrame. As discussed in the report, we are only going to keep the full hours values. We defined a function that will drop every 15 minute interval other then the 00 one. The goal is also to prefix column names from the imported datasets(masterssets per bidding zone) with the key to avoid collisions. 

In [17]:
import os

In [ ]:
def flatten_all_data(country_data):
    """Converts all Series/DataFrames in all_data[code] into a single wide DataFrame."""
    frames = []

    for key, df in country_data.items():
        if df is None:
            continue
        if isinstance(df, pd.Series):
            df = df.to_frame(name=key)
        else:
            df = df.copy()
            df.columns = [f"{key}_{col}" for col in df.columns]

        df = df[df.index.minute == 0]

        frames.append(df)

    if not frames:
        return None

    combined = pd.concat(frames, axis=1)
    combined.index.name = 'timestamp_naive'
    return combined


Combining the mastersets with country codes and loading up the file path

In [20]:
file_map = {
    'FR':    'FR_masterset_enriched.csv',
    'DE_LU': 'DE_LU_masterset_enriched.csv',
    'ES':    'ES_masterset_enriched.csv',
    'DK_1':  'DK1_masterset_enriched.csv',
    'NO_2':  'NO2_masterset_enriched.csv',
}

data_folder = r"C:\Users\User\OneDrive\Radna površina\Data Science Lab\Data"


This function merges freshly fetched country data into existing master CSV files.
For each country, loads the master CSV, flattens the fetched data into
a wide DataFrame, aligns both on a UTC-normalized timestamp index, and
left-joins the new columns onto the master while preserving all existing rows.
The result overwrites the original CSV file.

In [25]:
def merge_with_masterset(all_data, file_map, data_folder):
    for code, filename in file_map.items():
        print(f"\nProcessing {code}...")

        # Load the CSV
        filepath = os.path.join(data_folder, filename)
        master = pd.read_csv(filepath, parse_dates=['timestamp_naive'])
        master = master.set_index('timestamp_naive')

        # Make index timezone-naive for joining
        master.index = pd.to_datetime(master.index, utc=True)
        master.index = master.index.tz_localize(None)


        # Flatten the fetched data
        fetched = flatten_all_data(all_data[code])
        if fetched is None:
            print(f"  ✗ No fetched data for {code}, skipping.")
            continue

        # Make fetched index timezone-naive for joining
        fetched.index = fetched.index.tz_convert('UTC')
        fetched.index = fetched.index.tz_localize(None)

        # Left join — keeps all rows from the masterset
        merged = master.join(fetched, how='left')

        # Save back to CSV
        out_path = os.path.join(data_folder, filename)  # overwrites original
        # or use a new name: filename.replace('.csv', '_updated.csv')
        merged.to_csv(out_path)
        print(f"  ✓ Saved {merged.shape[0]} rows x {merged.shape[1]} cols → {filename}")



In [26]:

merge_with_masterset(all_data, file_map, data_folder)


Processing FR...


C:\Users\User\AppData\Local\Temp\ipykernel_20364\281619982.py:24: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  combined = pd.concat(frames, axis=1)


  ✓ Saved 28389 rows x 14 cols → FR_masterset_enriched.csv

Processing DE_LU...


C:\Users\User\AppData\Local\Temp\ipykernel_20364\281619982.py:24: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  combined = pd.concat(frames, axis=1)


  ✓ Saved 28381 rows x 13 cols → DE_LU_masterset_enriched.csv

Processing ES...


C:\Users\User\AppData\Local\Temp\ipykernel_20364\281619982.py:24: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  combined = pd.concat(frames, axis=1)


  ✓ Saved 28376 rows x 14 cols → ES_masterset_enriched.csv

Processing DK_1...
  ✓ Saved 28373 rows x 13 cols → DK1_masterset_enriched.csv

Processing NO_2...


C:\Users\User\AppData\Local\Temp\ipykernel_20364\281619982.py:24: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  combined = pd.concat(frames, axis=1)


  ✓ Saved 28378 rows x 14 cols → NO2_masterset_enriched.csv
